# CNN: 3 demos visuales para clase

Este notebook reúne 3 demos cortas y muy didácticas para explicar redes neuronales convolutivas:

1. **Convolución y detección de bordes**
2. **Entrenamiento de una CNN con MNIST**
3. **Visualización de feature maps**

Está pensado para usarse en Google Colab.


## Demo 1 — Convolución y detección de bordes

La idea es mostrar visualmente qué hace un filtro convolucional.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from skimage import data

# Imagen de ejemplo en escala de grises
img = data.camera()

# Kernel simple para detección de bordes
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])

# Aplicar convolución
filtered = convolve2d(img, kernel, mode="same", boundary="symm")

# Mostrar resultados
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.title("Imagen original")
plt.imshow(img, cmap="gray")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Kernel")
plt.imshow(kernel, cmap="gray")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Filtro de bordes")
plt.imshow(filtered, cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()


### Idea para explicar en clase

- La convolución toma una pequeña matriz llamada **kernel**.
- Ese kernel recorre la imagen.
- En cada posición calcula una nueva salida.
- En este caso, el filtro resalta los **bordes**.

Conecta muy bien con la idea de que una CNN aprende automáticamente filtros parecidos a estos.


## Demo 2 — Entrenamiento de una CNN con MNIST

Aquí mostramos una CNN sencilla para clasificar dígitos escritos a mano.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Cargar dataset MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalizar
x_train = x_train / 255.0
x_test = x_test / 255.0

# Agregar canal
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)


In [ ]:
# Mostrar algunas imágenes
plt.figure(figsize=(10, 3))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title(f"Etiqueta: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Definir modelo CNN
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()


In [ ]:
# Compilar modelo
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# Entrenar modelo
history = model.fit(
    x_train, y_train,
    epochs=5,
    validation_data=(x_test, y_test),
    verbose=1
)


In [ ]:
# Evaluar
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy en test: {accuracy:.4f}")


In [ ]:
# Graficar evolución del entrenamiento
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Pérdida")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Mostrar predicciones sobre algunas imágenes
predicciones = model.predict(x_test[:10])
clases_predichas = np.argmax(predicciones, axis=1)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"Pred: {clases_predichas[i]}\nReal: {y_test[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


### Idea para explicar en clase

- **Conv2D** aprende filtros.
- **MaxPooling** reduce tamaño y conserva lo importante.
- **Flatten** convierte mapas en vector.
- **Dense** toma la decisión final.

Flujo:

`imagen → convolución → pooling → flatten → dense → clase`


## Demo 3 — Visualización de feature maps

Aquí mostramos qué activaciones produce la CNN en sus primeras capas.


In [ ]:
from tensorflow.keras.models import Model

# Crear modelo intermedio para obtener salidas de capas
layer_outputs = [layer.output for layer in model.layers[:4]]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

# Elegir una imagen de prueba
img = x_test[0].reshape(1, 28, 28, 1)

# Obtener activaciones
activations = activation_model.predict(img)

print("Número de bloques visualizados:", len(activations))


In [ ]:
# Visualizar feature maps de la primera capa convolucional
first_layer = activations[0]

plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(first_layer[0, :, :, i], cmap="gray")
    plt.axis("off")
plt.suptitle("Feature maps - primera capa convolucional", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Visualizar feature maps de la segunda capa convolucional
second_conv = activations[2]

plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(second_conv[0, :, :, i], cmap="gray")
    plt.axis("off")
plt.suptitle("Feature maps - segunda capa convolucional", fontsize=14)
plt.tight_layout()
plt.show()


### Idea para explicar en clase

- Cada filtro responde a patrones distintos.
- En capas tempranas suelen aparecer bordes, trazos y contrastes.
- En capas más profundas aparecen patrones más complejos.

Esta demo ayuda mucho a que la CNN deje de sentirse como una caja negra.


## Cierre sugerido para la clase

Puedes cerrar con una idea como esta:

> Las CNN no ven la imagen como un humano. Van transformándola paso a paso en patrones cada vez más útiles para clasificarla.

Y luego conectar con:

- **CNN simple** → MNIST
- **CNN más profundas** → VGG, ResNet, MobileNet
- **Transfer learning** → modelos preentrenados
